<a href="https://colab.research.google.com/github/werowe/HypatiaAcademy/blob/master/ml/pythorch_linear_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# Building a Simple Linear Regression Model with PyTorch

This notebook demonstrates how to build and train a basic linear regression model using PyTorch. We'll cover the essential components: data preparation, model definition, loss function, optimizer, and the training loop.

Our goal is to train a model that can learn the relationship `y = 3x` from generated data.
```

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

torch.manual_seed(0) # For reproducibility of random initializations of weights and biases
np.set_printoptions(suppress=True) # Suppress scientific notation for NumPy array output for cleaner display

```markdown
## 1. Data Preparation

We'll create a simple dataset where the target `y` is a linear function of the input `x` (`y = 3x`). This makes it easy for our model to learn the underlying relationship.

First, we generate `x` and `y` using NumPy, and then convert them into PyTorch tensors, which are required for PyTorch operations.
```

In [10]:
# Data Generation (similar to a NumPy version)
# Create input (x) and target (y) data
x_np = np.arange(12).reshape(-1, 1).astype(np.float32) # Input features from 0 to 11. Reshape to a column vector.
y_np = 3 * x_np # Target values (y = 3x). This is the true relationship the model will try to learn.

# Convert NumPy arrays to PyTorch tensors
# PyTorch models operate on tensors, so this conversion is a crucial step.
x = torch.from_numpy(x_np)
y = torch.from_numpy(y_np)

print("Input (x) tensor:\n", x)
print("\nTarget (y) tensor:\n", y)

Input (x) tensor:
 tensor([[ 0.],
        [ 1.],
        [ 2.],
        [ 3.],
        [ 4.],
        [ 5.],
        [ 6.],
        [ 7.],
        [ 8.],
        [ 9.],
        [10.],
        [11.]])

Target (y) tensor:
 tensor([[ 0.],
        [ 3.],
        [ 6.],
        [ 9.],
        [12.],
        [15.],
        [18.],
        [21.],
        [24.],
        [27.],
        [30.],
        [33.]])


```markdown
## 2. Model Definition

We define a simple linear regression model using PyTorch's `nn.Module`. For linear regression, a single `nn.Linear` layer is sufficient. This layer performs a linear transformation on the input data: `y_pred = x * w + b`, where `w` is the weight and `b` is the bias.

*   `nn.Linear(in_features, out_features)`: Creates a linear transformation layer.
    *   `in_features=1`: Our input `x` has one feature.
    *   `out_features=1`: Our output `y` is a single value.
```

In [11]:
# Define the Linear Regression Model
# nn.Linear(input_features, output_features)
# Here, we have a single input feature (x) and we want a single output feature (y).
model = nn.Linear(1, 1)

print("Initial model parameters (weights and bias, randomly initialized):")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name}: {param.data.numpy()}")

Initial model parameters (weights and bias, randomly initialized):
weight: [[-0.00748682]]
bias: [0.5364436]


```markdown
## 3. Loss Function and Optimizer

To train our model, we need:

*   **A Loss Function**: This measures how far off our model's predictions are from the true target values. For regression tasks, Mean Squared Error (MSE) is a common choice.
    *   `nn.MSELoss()`: Calculates the mean of the squared differences between predictions and actual values.
*   **An Optimizer**: This algorithm updates the model's parameters (weights and bias) based on the loss, aiming to minimize it. Stochastic Gradient Descent (SGD) is a fundamental optimizer.
    *   `optim.SGD(model.parameters(), lr)`: Initializes SGD with the model's learnable parameters and a specified learning rate.
        *   `model.parameters()`: Provides all the parameters (weights and biases) of our model to the optimizer.
        *   `lr=0.01`: The learning rate determines the step size at each iteration while moving toward a minimum of the loss function.
```

In [12]:
# Define the Loss Function and Optimizer
criterion = nn.MSELoss() # Mean Squared Error loss is standard for regression problems.
optimizer = optim.SGD(model.parameters(), lr=0.01) # Stochastic Gradient Descent optimizer.
                                                 # It updates the model's weights and biases to reduce the loss.
                                                 # lr (learning rate) controls how big of a step the optimizer takes.

```markdown
## 4. Training Loop

This is the core of the model training. We iterate through a fixed number of `epochs` (training iterations). In each epoch:

1.  **Forward Pass**: The input `x` is passed through the model to get predictions `outputs`.
2.  **Calculate Loss**: The `criterion` (MSE) calculates the difference between `outputs` and the true `y`.
3.  **Zero Gradients**: Before backpropagation, we clear any previously calculated gradients to prevent accumulation.
4.  **Backward Pass (Backpropagation)**: `loss.backward()` computes the gradients of the loss with respect to each model parameter. These gradients indicate how much each parameter contributes to the error.
5.  **Optimize**: `optimizer.step()` updates the model's parameters using the calculated gradients and the learning rate.
```

In [13]:
# Training Loop
epochs = 500 # Number of times we'll iterate over the entire dataset.

print("\nStarting training...")
for epoch in range(epochs):
    # 1. Forward pass: Compute predicted y by passing x to the model
    outputs = model(x)

    # 2. Calculate the loss between predicted outputs and true y
    loss = criterion(outputs, y)

    # 3. Backward + optimize: Backpropagate the loss and update model parameters
    optimizer.zero_grad() # Clear previous gradients before computing new ones. Important!
    loss.backward()       # Compute gradients of the loss with respect to model parameters.
    optimizer.step()      # Update model parameters (weights and bias) using the optimizer's rules.

    # Print progress every 50 epochs
    if (epoch+1) % 50 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

print("Training complete.")
print("\nFinal model parameters:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name}: {param.data.numpy()}")


Starting training...
Epoch [50/500], Loss: 0.1372
Epoch [100/500], Loss: 0.0786
Epoch [150/500], Loss: 0.0450
Epoch [200/500], Loss: 0.0258
Epoch [250/500], Loss: 0.0148
Epoch [300/500], Loss: 0.0085
Epoch [350/500], Loss: 0.0048
Epoch [400/500], Loss: 0.0028
Epoch [450/500], Loss: 0.0016
Epoch [500/500], Loss: 0.0009
Training complete.

Final model parameters:
weight: [[2.9925888]]
bias: [0.05644495]


```markdown
## 5. Predictions and Results

After training, we can use our model to make predictions on the input data `x` and compare them with the original target `y` values. We use `torch.no_grad()` to temporarily disable gradient calculations, as we are only inferring, not training.
```

In [14]:
# Make Predictions with the trained model
# We use torch.no_grad() because we don't need to compute gradients during inference.
with torch.no_grad():
    predictions = model(x).numpy() # Get predictions from the model and convert to NumPy array for easy comparison.

# Display Results
print("\nOriginal Input (x), True Target (y), and Predicted (y_pred) values:")
print(np.column_stack([x_np, y_np, predictions])) # Stack arrays horizontally for clear display.

# You can see that the predicted values are very close to the true target values (y_np).
# For example, for x=1, y=3, and prediction is close to 3.
# This indicates that our simple linear regression model has successfully learned the relationship y = 3x.


Original Input (x), True Target (y), and Predicted (y_pred) values:
[[ 0.          0.          0.05644495]
 [ 1.          3.          3.0490336 ]
 [ 2.          6.          6.0416226 ]
 [ 3.          9.          9.034211  ]
 [ 4.         12.         12.0268    ]
 [ 5.         15.         15.019389  ]
 [ 6.         18.         18.011976  ]
 [ 7.         21.         21.004566  ]
 [ 8.         24.         23.997154  ]
 [ 9.         27.         26.989742  ]
 [10.         30.         29.982332  ]
 [11.         33.         32.974922  ]]


In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

torch.manual_seed(0) # For reproducibility of random initializations
np.set_printoptions(suppress=True) # Suppress scientific notation for NumPy arrays

# Data (same as your NumPy version) - Create input (x) and target (y) data
x_np = np.arange(12).reshape(-1, 1).astype(np.float32) # Input features from 0 to 11
y_np = 3 * x_np # Target values (y = 3x)

# Convert to PyTorch tensors from NumPy arrays
x = torch.from_numpy(x_np)
y = torch.from_numpy(y_np)


model = nn.Linear(1, 1) # A single input feature, a single output feature

# Loss and optimizer (same as Keras) - Define the loss function and optimizer
criterion = nn.MSELoss() # Mean Squared Error loss for regression
optimizer = optim.SGD(model.parameters(), lr=0.01) # Stochastic Gradient Descent optimizer with a learning rate of 0.01

# Training loop - Iterate to train the model
epochs = 500 # Number of training iterations
for epoch in range(epochs):
    # Forward pass: Compute predicted y by passing x to the model
    outputs = model(x)
    # Calculate the loss between predicted outputs and true y
    loss = criterion(outputs, y)

    # Backward + optimize: Backpropagate the loss and update model parameters
    optimizer.zero_grad() # Clear previous gradients
    loss.backward() # Compute gradients of the loss with respect to model parameters
    optimizer.step() # Update model parameters using the optimizer

# Predictions - Make predictions with the trained model
with torch.no_grad(): # Disable gradient calculation for inference
    predictions = model(x).numpy() # Get predictions and convert back to NumPy array

# Results - Print the original, true, and predicted values
print("\nOriginal vs Predicted:")
print(np.column_stack([x_np, y_np, predictions])) # Stack arrays horizontally for display


Original vs Predicted:
[[ 0.          0.          0.05644495]
 [ 1.          3.          3.0490336 ]
 [ 2.          6.          6.0416226 ]
 [ 3.          9.          9.034211  ]
 [ 4.         12.         12.0268    ]
 [ 5.         15.         15.019389  ]
 [ 6.         18.         18.011976  ]
 [ 7.         21.         21.004566  ]
 [ 8.         24.         23.997154  ]
 [ 9.         27.         26.989742  ]
 [10.         30.         29.982332  ]
 [11.         33.         32.974922  ]]


In [16]:
# This cell was part of the original notebook, but its content has been redistributed
# into new cells with detailed explanations and markdown descriptions for clarity.
# It is now empty to avoid redundant code execution.